In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from dotenv import load_dotenv

load_dotenv()


# Inicializa a sessão do Spark com os drivers do Kafka e do S3
spark = SparkSession.builder \
    .appName("Security-DataLake-Pipeline") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.apache.hadoop:hadoop-aws:3.3.4") \
    .getOrCreate()

# --- CONFIGURAÇÃO DE AUTENTICAÇÃO DO GARAGE S3 ---
# Usando a Access Key que geramos no CLI do Garage
access_key = os.getenv('minha_chave')
# ⚠️ Substitua abaixo pelo Secret Key que o comando 'key create' exibiu no seu terminal
secret_key = os.getenv('secret_key')

sc = spark.sparkContext
sc._jsc.hadoopConfiguration().set("fs.s3a.access.key", access_key)
sc._jsc.hadoopConfiguration().set("fs.s3a.secret.key", secret_key)

# Endereço interno do contêiner do Garage na rede do Docker
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://garage:3900")
sc._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

print("🚀 Sessão Spark configurada com sucesso para o Garage HQ e Kafka!")

In [ ]:
# Criando um pequeno DataFrame de testes com alertas fictícios
dados_teste = [
    ("VLAN_30", "192.168.30.99", "Mirai Port Scan Detectado", "ALTA"),
    ("VLAN_40", "192.168.40.12", "Tentativa Bruteforce SSH", "MEDIA")
]

colunas = ["origem_vlan", "ip_origem", "evento", "severidade"]
df_teste = spark.createDataFrame(dados_teste, schema=colunas)

# Tentando persistir em formato colunar Parquet no Garage S3
try:
    df_teste.write \
        .format("parquet") \
        .mode("overwrite") \
        .save("s3a://meu-data-lake/teste_alertas/")
    print("✨ Vitória! O Spark gravou o arquivo Parquet com sucesso no Garage HQ.")
except Exception as e:
    print(f"❌ Falha ao gravar no Object Storage: {e}")

In [ ]:
# 1. Definindo o Schema esperado para os logs brutos de rede (Baseado em NetFlow/OCSF)
log_schema = StructType([
    StructField("vlan", IntegerType(), True),
    StructField("src_ip", StringType(), True),
    StructField("dst_ip", StringType(), True),
    StructField("dst_port", IntegerType(), True),
    StructField("packets", IntegerType(), True),
    StructField("bytes", IntegerType(), True)
])

# 2. Conectando o Spark Streaming ao broker interno do Kafka (Porta 9092)
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "security-network-flux") \
    .option("startingOffsets", "latest") \
    .load()

# 3. Tratando os dados: Convertendo o binário do Kafka em String/JSON estruturado
df_parsed = df_kafka.selectExpr("CAST(value AS STRING) as json_payload") \
    .select(from_json(col("json_payload"), log_schema).alias("data")) \
    .select("data.*") \
    .withColumn("ingestion_time", current_timestamp())

# 4. Gravando o fluxo contínuo no Garage particionado por VLAN (Otimiza futuras consultas de ML)
query = df_parsed.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", "s3a://meu-data-lake/live_network_logs/") \
    .option("checkpointLocation", "s3a://meu-data-lake/checkpoints/logs_pipeline/") \
    .partitionBy("vlan") \
    .start()

print("🔥 Pipeline em tempo real ativado e escutando o Kafka...")